# AgriVision: MODIS + Sentinel-2 fusion pipeline (synthetic sample data)

This notebook follows **page 3 of `docs/diagrams/AgriVision_architecture.drawio`** box by box:

| Diagram box | Notebook section |
|---|---|
| Module 1 · Data (GEE) | **M1** synthetic Sentinel-2 L2A, MODIS MOD09GQ, MODIS MOD09GA |
| Module 2 · Preprocessing (reflectance kept, no NDVI) | **M2** S2 branch: 2.1 cloud mask → 2.2 common UTM grid → 2.3 co-registration; MODIS branch: 2.4 cloud + view-angle mask → 2.5 bilinear resample → 2.6 PSF + offset fit → 2.7 band adjustment |
| Module 3 · Dataset | **M3** samples (C1, F1, C2) → F2 |
| Module 4 · Fusion network | **M4** fine encoder, change encoder, fusion, decoder |
| Modules 5–6 · Loss & training | **M5–6** |
| Module 7 · Inference → Optional per-band blend → Predicted image | **M7** |
| Module 8 · Evaluation | **M8** |
| Module 9 · NDVI & crop monitoring (LAST STEP) | **M9** |

**Why synthetic data?** The preprocessing lead's real exports are not available yet. Module 1 therefore
*simulates* what Google Earth Engine would deliver: the same products, bands, scale factors and quality
layers, with realistic problems built in (clouds, cloud shadows, Sentinel-2 misregistration, MODIS's
231.66 m native grid, MODIS blur that grows with view angle, MODIS geolocation error, and MODIS/S2 band
differences). Every later module only sees what a real export would contain, so Modules 2–9 work
unchanged when real data arrives. Because we generated the scene, we also know the hidden "truth", and
the notebook uses it to check that each step really fixes the problem it is meant to fix.

**Running it.** In Colab: *Runtime → Change runtime type → GPU*, then *Run all*. Locally (e.g. an
RTX 5060): any Jupyter with PyTorch ≥ 2.7 built for CUDA 12.8 (needed for RTX 50-series cards).
A full run takes roughly 10–20 minutes on a GPU.

## 0 · Setup
Installs the two packages Colab does not ship with, picks the GPU, and fixes all random seeds so the
synthetic scene is identical every run.

In [ ]:
%pip install -q rasterio pytorch-msssim

In [ ]:
import os, math, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, map_coordinates, binary_dilation, shift as nd_shift
from scipy.signal import savgol_filter
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity
import rasterio
from rasterio.transform import from_origin
from IPython.display import display
import torch, torch.nn as nn, torch.nn.functional as F
from pytorch_msssim import ms_ssim

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

# FAST = 1 shrinks training so the notebook can be smoke-tested on a CPU. Leave it at 0.
FAST = os.environ.get('AGRIVISION_FAST', '0') == '1'

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '| device:', DEV)
if DEV == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    major, minor = torch.cuda.get_device_capability(0)
    if (major, minor) >= (12, 0) and 'sm_120' not in torch.cuda.get_arch_list():
        print('WARNING: this PyTorch build has no kernels for RTX 50-series (sm_120). Install a CUDA 12.8 '
              'build: pip install torch --index-url https://download.pytorch.org/whl/cu128')

# Where outputs go: Google Drive in Colab (survives disconnects), a local folder elsewhere.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/agrivision_synthetic'
except ImportError:
    ROOT = os.path.abspath('agrivision_synthetic')
PRED_DIR, CKPT_DIR = f'{ROOT}/predicted', f'{ROOT}/checkpoints'
for d in (ROOT, PRED_DIR, CKPT_DIR):
    os.makedirs(d, exist_ok=True)
print('outputs →', ROOT)

### Scene and grid configuration
* **Fine grid** = Sentinel-2's 10 m grid: 750 × 750 pixels (7.5 km × 7.5 km) in UTM zone 43N.
* **Common coarse grid** = 250 m, sharing the fine grid's origin, so one cell = 25 × 25 fine pixels.
  This makes the *files* line up. It does not make MODIS data line up; that is Module 2's job.
* **MODIS native grid** = 231.656 m cells with its own origin (the real MODIS "250 m" grid).
* Season: 150 days from 1 Nov 2023, Sentinel-2 every 5 days, MODIS every day.

In [ ]:
H = W = 750
RES_F, RES_C = 10.0, 250.0
K = int(RES_C / RES_F)                      # 25 fine pixels per common coarse cell
HC, WC = H // K, W // K                     # 30 × 30 common grid
MODIS_RES = 231.656358                      # MOD09GQ native cell size (m)
MODIS_1KM = 4 * MODIS_RES                   # MOD09GA state_1km cell size (m)
X0, Y0 = 500000.0, 3500000.0                # upper-left corner of the AOI (UTM 43N, metres)
CRS = 'EPSG:32643'
FINE_TRANSFORM = from_origin(X0, Y0, RES_F, RES_F)
SEASON_START, N_DAYS, S2_REVISIT = dt.date(2023, 11, 1), 150, 5
BANDS = ['red', 'nir']
day2date = lambda d: SEASON_START + dt.timedelta(days=int(d))

---
# Module 1 · Data (GEE) — synthetic sample data
Real use: export from Google Earth Engine (see the guide, `docs/fusion_model_modules.md`, M1).
Here we build a farm landscape and simulate the three products the diagram lists:

| Product | Bands | What we simulate |
|---|---|---|
| **Sentinel-2 L2A** | B4 red, B8 NIR, Cloud Score+ `cs_cdf` | 10 m, every 5 days, clouds + shadows, sub-pixel misregistration, sensor noise, stored ×10 000 like GEE |
| **MODIS MOD09GQ** | `sur_refl_b01` red, `sur_refl_b02` NIR | daily, native 231.66 m grid, blur that grows with view angle, ~50 m geolocation error, different spectral response, clouds, stored ×10 000 |
| **MODIS MOD09GA** | `state_1km`, `SensorZenith` | 1 km cloud/shadow/adjacency bits and view angle (×100), as in the real product |

### 1a · Fields and land cover
Voronoi polygons stand in for field parcels. Each field gets a land-cover class; two roads cut through
the scene. In real use the field boundaries would come from your field polygon layer.

In [ ]:
CLASSES = {'wheat': 0.45, 'mustard': 0.22, 'fallow': 0.13, 'orchard': 0.12, 'water': 0.08}

def make_fields(n_fields=60):
    seeds = rng.uniform(0, [H, W], size=(n_fields, 2))
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
    lab = np.zeros((H, W), np.int32)
    best = np.full((H, W), np.inf, np.float32)
    for i, (sy, sx) in enumerate(seeds):
        d = (yy - sy) ** 2 + (xx - sx) ** 2
        m = d < best
        best[m], lab[m] = d[m], i
    return lab

field_id = make_fields()
N_FIELDS = field_id.max() + 1
names = list(CLASSES)
field_class = rng.choice(names, size=N_FIELDS, p=list(CLASSES.values()))
field_class[rng.integers(N_FIELDS)] = 'water'          # at least one pond

road = np.zeros((H, W), bool)
road[310:313, :] = True
road[:, 455:458] = True
ROAD_ID = -1
field_id[road] = ROAD_ID

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
cls_map = np.where(road, -1, np.vectorize(names.index)(field_class[np.clip(field_id, 0, None)]))
ax.imshow(cls_map, cmap='tab10', interpolation='nearest')
ax.set_title('Land cover (roads = darkest)'); ax.axis('off'); plt.show()
print({c: int((field_class == c).sum()) for c in names}, 'fields')

### 1b · Crop phenology and true surface reflectance
Each field follows a seasonal **vegetation-cover curve** (a double logistic: green-up, then senescence)
with its own sowing and harvest dates. Reflectance is a mix of soil and green vegetation, so red falls
and NIR rises as the crop grows. Fields also vary inside themselves (patchy growth).

We also inject **crop stress** in three wheat fields: from a given day, a circular patch loses up to
40 % of its green cover. Module 9 should find these, and we keep the answer key (`STRESS`).

In [ ]:
def double_logistic(t, fmin, fmax, sos, eos, r_up, r_down):
    return fmin + (fmax - fmin) * (1 / (1 + np.exp(-(t - sos) / r_up)) - 1 / (1 + np.exp(-(t - eos) / r_down)))

days = np.arange(N_DAYS)
fv_table = np.zeros((N_FIELDS, N_DAYS), np.float32)        # green cover per field per day
for i, c in enumerate(field_class):
    if c == 'wheat':
        fv_table[i] = double_logistic(days, 0.04, rng.uniform(0.82, 0.95), rng.uniform(30, 42),
                                      rng.uniform(118, 132), 6, 5)
    elif c == 'mustard':
        fv_table[i] = double_logistic(days, 0.04, rng.uniform(0.70, 0.85), rng.uniform(15, 25),
                                      rng.uniform(88, 100), 5, 5)
    elif c == 'orchard':
        fv_table[i] = rng.uniform(0.5, 0.6) + 0.03 * np.sin(days / 25)
    elif c == 'fallow':
        fv_table[i] = 0.05 + 0.04 * np.exp(-((days - 70) / 20) ** 2)   # a little weed growth
fv_table = np.clip(fv_table, 0, 1)

soil_red = rng.uniform(0.17, 0.24, N_FIELDS).astype(np.float32)        # soil brightness per field
heterog = 1 + 0.08 * gaussian_filter(rng.standard_normal((H, W)), 12) / 0.02
heterog = np.clip(heterog, 0.8, 1.15).astype(np.float32)               # patchy growth inside fields

wheat_ids = np.flatnonzero(field_class == 'wheat')
STRESS = []
for fid in rng.choice(wheat_ids, size=3, replace=False):
    ys, xs = np.nonzero(field_id == fid)
    cy, cx = ys.mean(), xs.mean()
    radius = rng.uniform(35, 60)
    start = int(rng.uniform(70, 95))
    STRESS.append(dict(field=int(fid), start_day=start, cy=cy, cx=cx, r=radius))
yy, xx = np.mgrid[0:H, 0:W]
stress_mask = np.zeros((H, W), bool)
for s in STRESS:
    s['mask'] = ((yy - s['cy']) ** 2 + (xx - s['cx']) ** 2 < s['r'] ** 2) & (field_id == s['field'])
    stress_mask |= s['mask']

def true_reflectance(day):
    """Noise-free 10 m (red, NIR) reflectance on a given day. Only the simulator may call this."""
    fid = np.clip(field_id, 0, None)
    fv = fv_table[fid, day] * heterog
    for s in STRESS:
        loss = 0.4 * np.clip((day - s['start_day']) / 12, 0, 1)
        fv = np.where(s['mask'], fv * (1 - loss), fv)
    fv = np.clip(fv, 0, 1)
    sr = soil_red[fid]
    red = fv * 0.03 + (1 - fv) * sr
    nir = fv * 0.48 + (1 - fv) * (1.3 * sr + 0.02)
    water = (field_class[fid] == 'water') & ~road
    red, nir = np.where(water, 0.04, red), np.where(water, 0.02, nir)
    red, nir = np.where(road, 0.16, red), np.where(road, 0.20, nir)
    return np.stack([red, nir]).astype(np.float32)

fig, axs = plt.subplots(1, 3, figsize=(13, 4))
for ax, d in zip(axs, [10, 60, 110]):
    r = true_reflectance(d)
    ax.imshow((r[1] - r[0]) / (r[1] + r[0]), vmin=0, vmax=0.9, cmap='RdYlGn'); ax.axis('off')
    ax.set_title(f'TRUE NDVI day {d} ({day2date(d)})\n(for checking only)')
plt.show()

### 1c · Sentinel-2 L2A simulator
Every 5 days. Each acquisition gets:
* **clouds** (smooth random blobs, from none to almost full cover) and their **shadows**;
* a **Cloud Score+ `cs_cdf`** layer: high for clear pixels, low for cloud and shadow;
* a random **sub-pixel misregistration** (up to ±1.5 px), as happens between real S2 dates;
* sensor noise, then storage as integers ×10 000 (exactly how GEE serves `S2_SR_HARMONIZED`).

In [ ]:
def cloud_field(frac, sigma=28):
    """Boolean cloud mask covering about `frac` of the scene."""
    if frac <= 0:
        return np.zeros((H, W), bool)
    n = gaussian_filter(rng.standard_normal((H, W)), sigma)
    return n > np.quantile(n, 1 - frac)

S2_DAYS = list(range(2, N_DAYS, S2_REVISIT))
s2_raw, cloud_by_day, shadow_by_day, S2_TRUE_SHIFT = {}, {}, {}, {}
for d in S2_DAYS:
    frac = rng.choice([0.0, rng.uniform(0.05, 0.3), rng.uniform(0.4, 0.95)], p=[0.55, 0.15, 0.30])
    cloud = cloud_field(frac)
    shadow = np.roll(cloud, (18, 22), axis=(0, 1)) & ~cloud
    cloud_by_day[d], shadow_by_day[d] = cloud, shadow
    shift_px = rng.uniform(-1.5, 1.5, 2)
    S2_TRUE_SHIFT[d] = shift_px
    refl = nd_shift(true_reflectance(d), (0, *shift_px), order=1, mode='nearest')
    refl = refl + rng.normal(0, 0.004, refl.shape)
    refl = np.where(shadow, refl * 0.45, refl)
    refl = np.where(cloud, np.stack([np.full((H, W), 0.38), np.full((H, W), 0.42)]), refl)
    cs = 0.95 - 0.9 * gaussian_filter(cloud.astype(np.float32), 2) - 0.6 * shadow
    cs = np.clip(cs + rng.normal(0, 0.03, (H, W)), 0, 1)
    s2_raw[d] = {'B4': np.clip(refl[0] * 1e4, 0, 1e4).astype(np.uint16),
                 'B8': np.clip(refl[1] * 1e4, 0, 1e4).astype(np.uint16),
                 'cs_cdf': cs.astype(np.float32)}
print(len(S2_DAYS), 'Sentinel-2 acquisitions; cloud fraction per date:',
      [round(cloud_by_day[d].mean(), 2) for d in S2_DAYS])

### 1d · MODIS MOD09GQ simulator (daily red + NIR)
For each day:
* **Native grid:** 231.656 m cells on their own origin, covering the AOI plus a margin.
* **Blur:** each cell sees a blurred patch of ground: the cell area plus the sensor's point-spread
  function, which **widens with view angle** (up to ~2.5× at the scan edge).
* **Geolocation error:** a systematic part (+3, −2 fine px ≈ 30 m, 20 m) plus a random daily part (~15 m).
* **Spectral difference:** MODIS bands differ from S2 B4/B8, so MODIS = g·S2 + o per band.
* **Clouds:** on S2 days the same clouds as S2 (both pass ~10:30); other days have their own clouds.
  Partly cloudy cells are contaminated in proportion to their cloud fraction.

In [ ]:
MODIS_SYS_ERR = np.array([3.0, -2.0])                 # systematic geolocation error (fine px: dy, dx)
MODIS_GAIN = np.array([0.94, 0.92]); MODIS_OFFSET = np.array([0.008, 0.015])
MARGIN_M = 700.0
mod_x0 = X0 - MARGIN_M - 37.3                          # MODIS grid origin (does not match the AOI)
mod_y0 = Y0 + MARGIN_M + 61.7
MOD_NX = int(np.ceil((W * RES_F + 2 * MARGIN_M) / MODIS_RES))
MOD_NY = int(np.ceil((H * RES_F + 2 * MARGIN_M) / MODIS_RES))

def cell_centres_in_fine_px(x0, y0, res, ny, nx):
    """Rows/cols (in fine-pixel units) of the centres of a grid's cells."""
    xc = x0 + (np.arange(nx) + 0.5) * res
    yc = y0 - (np.arange(ny) + 0.5) * res
    col = (xc - X0) / RES_F - 0.5
    row = (Y0 - yc) / RES_F - 0.5
    return np.meshgrid(row, col, indexing='ij')

MOD_ROWS, MOD_COLS = cell_centres_in_fine_px(mod_x0, mod_y0, MODIS_RES, MOD_NY, MOD_NX)
BOX_SIGMA = MODIS_RES / RES_F / np.sqrt(12)            # a 231 m box ≈ Gaussian of this width

mod09gq, MODIS_TRUE = {}, {}
for d in range(N_DAYS):
    vz = rng.uniform(0, 55)                             # sensor view zenith angle (deg)
    psf = 7.0 * (1 + 1.5 * (vz / 55) ** 2)              # PSF width (fine px), grows off-nadir
    sigma = np.hypot(BOX_SIGMA, psf)
    err = MODIS_SYS_ERR + rng.normal(0, 1.5, 2)
    if d in cloud_by_day:
        cloud, shadow = cloud_by_day[d], shadow_by_day[d]
    else:
        cloud = cloud_field(rng.choice([0.0, rng.uniform(0.05, 0.3), rng.uniform(0.4, 0.95)],
                                       p=[0.55, 0.2, 0.25]))
        shadow = np.roll(cloud, (18, 22), axis=(0, 1)) & ~cloud
    rows, cols = MOD_ROWS + err[0], MOD_COLS + err[1]
    truth = true_reflectance(d)
    cell = np.stack([map_coordinates(gaussian_filter(b, sigma, mode='nearest'), [rows, cols],
                                     order=1, mode='nearest') for b in truth])
    cfrac = map_coordinates(gaussian_filter(cloud.astype(np.float32), sigma), [rows, cols], order=1,
                            mode='nearest')
    sfrac = map_coordinates(gaussian_filter(shadow.astype(np.float32), sigma), [rows, cols], order=1,
                            mode='nearest')
    modis = MODIS_GAIN[:, None, None] * cell + MODIS_OFFSET[:, None, None]
    modis = modis * (1 - 0.55 * sfrac) * (1 - cfrac) + np.array([0.38, 0.42])[:, None, None] * cfrac
    modis = modis + rng.normal(0, 0.006, modis.shape)
    mod09gq[d] = {'sur_refl_b01': np.round(modis[0] * 1e4).astype(np.int16),
                  'sur_refl_b02': np.round(modis[1] * 1e4).astype(np.int16)}
    MODIS_TRUE[d] = dict(vz=vz, err=err, sigma=sigma, cloud=cloud, shadow=shadow)
print('MOD09GQ native grid:', MOD_NY, '×', MOD_NX, 'cells of', round(MODIS_RES, 2), 'm, for', N_DAYS, 'days')

### 1e · MODIS MOD09GA simulator (`state_1km`, `SensorZenith`)
Real MOD09GQ has **no cloud flags**, so the pipeline takes them from MOD09GA on the same day.
`state_1km` bits used here, as in the real product: bits 0–1 cloud state (00 clear, 01 cloudy,
10 mixed), bit 2 cloud shadow, bit 13 pixel adjacent to cloud. `SensorZenith` is degrees × 100.

In [ ]:
GA_NX, GA_NY = int(np.ceil(MOD_NX / 4)), int(np.ceil(MOD_NY / 4))
GA_ROWS, GA_COLS = cell_centres_in_fine_px(mod_x0, mod_y0, MODIS_1KM, GA_NY, GA_NX)

mod09ga = {}
for d in range(N_DAYS):
    t = MODIS_TRUE[d]
    k = MODIS_1KM / RES_F / 2
    cf = map_coordinates(gaussian_filter(t['cloud'].astype(np.float32), k / 1.7), [GA_ROWS, GA_COLS],
                         order=1, mode='nearest')
    sf = map_coordinates(gaussian_filter(t['shadow'].astype(np.float32), k / 1.7), [GA_ROWS, GA_COLS],
                         order=1, mode='nearest')
    state = np.zeros(cf.shape, np.uint16)
    state[cf > 0.6] |= 0b01                       # cloudy
    state[(cf > 0.02) & (cf <= 0.6)] |= 0b10      # mixed
    state[sf > 0.3] |= 1 << 2                     # cloud shadow
    near = binary_dilation(cf > 0.02, iterations=1) & ~(cf > 0.02)
    state[near] |= 1 << 13                        # adjacent to cloud
    zen = t['vz'] + 0.5 * np.linspace(-1, 1, GA_NX)[None, :] + 0 * cf
    mod09ga[d] = {'state_1km': state, 'SensorZenith': np.round(zen * 100).astype(np.int16)}
print('MOD09GA 1 km grid:', GA_NY, '×', GA_NX)

### 1f · Quick look at the raw "exports"
What the preprocessing lead would hand over. Note the clouds, the coarse MODIS, and that nothing here
is NDVI: these are raw bands.

In [ ]:
d_show = S2_DAYS[6]
fig, axs = plt.subplots(1, 4, figsize=(17, 4))
axs[0].imshow(s2_raw[d_show]['B8'], cmap='gray'); axs[0].set_title(f'S2 B8 (DN) day {d_show}')
axs[1].imshow(s2_raw[d_show]['cs_cdf'], vmin=0, vmax=1); axs[1].set_title('S2 Cloud Score+ cs_cdf')
axs[2].imshow(mod09gq[d_show]['sur_refl_b02'], cmap='gray'); axs[2].set_title('MOD09GQ b02 (native grid)')
axs[3].imshow(mod09ga[d_show]['state_1km'] & 3, vmin=0, vmax=2); axs[3].set_title('MOD09GA cloud state bits')
for a in axs: a.axis('off')
plt.show()

---
# Module 2 · Preprocessing (reflectance kept, no NDVI)
Two branches, exactly as in the diagram. **Output:** red and NIR *reflectance* for both sensors on one
10 m grid. NDVI is not computed here.

## Sentinel-2 branch
### 2.1 · Cloud mask (Cloud Score+ ≥ 0.65, dilate ~3 px)
Scale DN to reflectance (÷10 000), mark pixels with `cs_cdf < 0.65` as invalid, and grow the invalid
area by 3 px to remove cloud edges, which Cloud Score+ tends to under-flag.

In [ ]:
CS_THRESHOLD, DILATE_PX = 0.65, 3

def s2_cloud_mask(raw):
    refl = np.stack([raw['B4'], raw['B8']]).astype(np.float32) / 1e4
    bad = binary_dilation(raw['cs_cdf'] < CS_THRESHOLD, iterations=DILATE_PX)
    refl[:, bad] = np.nan
    return refl

s2_masked = {d: s2_cloud_mask(s2_raw[d]) for d in S2_DAYS}
s2_clear_frac = {d: float(np.isfinite(s2_masked[d][0]).mean()) for d in S2_DAYS}
missed = [float((cloud_by_day[d] & np.isfinite(s2_masked[d][0])).sum()) for d in S2_DAYS]
print('clear fraction per date:', {d: round(v, 2) for d, v in s2_clear_frac.items()})
print('check: cloudy pixels the mask let through (all dates):', int(sum(missed)))

### 2.2 · Common UTM grid (10 m)
Puts every image on the shared grid (EPSG:32643, origin `X0, Y0`, 10 m). With real data GEE does this
at export (`crs`, `crsTransform`). Our synthetic S2 already sits on it, so this step only checks that
the grids match; the general resampler is used for MODIS in step 2.5.

In [ ]:
def to_common_fine_grid(arr, transform):
    assert transform == FINE_TRANSFORM and arr.shape[-2:] == (H, W), 'grid mismatch: reproject first'
    return arr

s2_grid = {d: to_common_fine_grid(s2_masked[d], FINE_TRANSFORM) for d in S2_DAYS}
print('S2 on common 10 m grid:', FINE_TRANSFORM)

### 2.3 · Co-registration (all dates vs one master date)
Even a 1-pixel shift between S2 dates creates false "change" at field edges, and the model would learn
it. Real pipeline: **AROSICS** (`COREG_LOCAL`). Here: the same idea, global phase correlation of the
NIR band against the clearest date, then shift each image back. Dates under 30 % clear are skipped
(too little to match; they are unusable for training anyway).

In [ ]:
MASTER = max(S2_DAYS, key=lambda d: s2_clear_frac[d])

def fill_nan(a):
    return np.where(np.isfinite(a), a, np.nanmean(a))

s2 = {}
coreg_err = []
for d in S2_DAYS:
    img = s2_grid[d]
    if d == MASTER or s2_clear_frac[d] < 0.3:
        s2[d] = img
        continue
    shift_est, _, _ = phase_cross_correlation(fill_nan(s2_grid[MASTER][1]), fill_nan(img[1]),
                                              upsample_factor=20)
    valid = np.isfinite(img[0]).astype(np.float32)
    moved = np.stack([nd_shift(np.nan_to_num(b), shift_est, order=1, mode='nearest') for b in img])
    moved[:, nd_shift(valid, shift_est, order=1, mode='constant') < 0.99] = np.nan
    s2[d] = moved
    coreg_err.append(np.abs(S2_TRUE_SHIFT[d] + shift_est - S2_TRUE_SHIFT[MASTER]).max())
print(f'master date: day {MASTER}; residual misregistration after co-registration: '
      f'mean {np.mean(coreg_err):.2f} px, worst {np.max(coreg_err):.2f} px (before: up to 3 px)')

## MODIS branch
### 2.4 · Cloud + view-angle mask (`state_1km`, zenith < 35–40°)
Decode the MOD09GA bits, carry each 1 km flag to the 231 m cells inside it, scale DN to reflectance,
and drop whole days when the sensor looked too far off-nadir (the footprint gets too large).

In [ ]:
MAX_ZENITH = 40.0
# position of each 231 m cell inside the 1 km grid (nearest 1 km cell)
GQ_TO_GA = (np.minimum(np.arange(MOD_NY)[:, None] // 4, GA_NY - 1),
            np.minimum(np.arange(MOD_NX)[None, :] // 4, GA_NX - 1))

def modis_cloud_angle_mask(gq, ga):
    state = ga['state_1km'][GQ_TO_GA]
    clear = ((state & 0b11) == 0) & ((state & (1 << 2)) == 0) & ((state & (1 << 13)) == 0)
    zenith = ga['SensorZenith'][GQ_TO_GA] / 100.0
    ok = clear & (zenith < MAX_ZENITH)
    refl = np.stack([gq['sur_refl_b01'], gq['sur_refl_b02']]).astype(np.float32) / 1e4
    refl[:, ~ok] = np.nan
    return refl

modis_native = {d: modis_cloud_angle_mask(mod09gq[d], mod09ga[d]) for d in range(N_DAYS)}
inner = (slice(4, -4), slice(4, -4))                   # cells over the AOI, not the margin
modis_valid = {d: float(np.isfinite(modis_native[d][0][inner]).mean()) for d in range(N_DAYS)}
print('days with ≥ 95 % valid MODIS over the AOI:', sum(v >= 0.95 for v in modis_valid.values()), 'of', N_DAYS)

### 2.5 · Bilinear resample (to common grid, upsample to 10 m)
MODIS cells (231.66 m, own origin) do not match our grids. Resample **bilinearly** (never nearest
neighbour, which would drop about one source cell in 13) to:
* the **common 250 m grid** (30 × 30), used to compare with S2 in 2.6 and 2.7;
* the **10 m grid** (750 × 750), the form the network takes.

Invalid (masked) cells stay invalid: a target pixel is kept only if all source cells it draws on are valid.
(Our synthetic MODIS grid is axis-aligned with UTM. Real MODIS is sinusoidal and needs a full
reprojection, e.g. `rasterio.warp.reproject(..., resampling=Resampling.bilinear)`, but the idea is the same.)

In [ ]:
def resample_bilinear(src, src_x0, src_y0, src_res, dst_x0, dst_y0, dst_res, dst_shape):
    """NaN-aware bilinear resampling between two axis-aligned grids (coordinates in metres)."""
    ny, nx = dst_shape
    xc = dst_x0 + (np.arange(nx) + 0.5) * dst_res
    yc = dst_y0 - (np.arange(ny) + 0.5) * dst_res
    cols = (xc - src_x0) / src_res - 0.5
    rows = (src_y0 - yc) / src_res - 0.5
    rr, cc = np.meshgrid(rows, cols, indexing='ij')
    valid = np.isfinite(src[0]).astype(np.float32)
    w = map_coordinates(valid, [rr, cc], order=1, mode='nearest')
    out = np.stack([map_coordinates(np.nan_to_num(b), [rr, cc], order=1, mode='nearest') for b in src])
    out[:, w < 0.999] = np.nan
    return out.astype(np.float32)

modis_common = {d: resample_bilinear(modis_native[d], mod_x0, mod_y0, MODIS_RES, X0, Y0, RES_C, (HC, WC))
                for d in range(N_DAYS)}

def modis_upsampled_10m(d):
    return resample_bilinear(modis_native[d], mod_x0, mod_y0, MODIS_RES, X0, Y0, RES_F, (H, W))

print('common grid:', modis_common[0].shape, '| 10 m:', modis_upsampled_10m(0).shape)

### 2.6 · PSF + offset fit (per clear same-day pair; shift MODIS back)
A MODIS cell is neither a sharp 250 m square nor exactly where its coordinates say. On every day where
**both** sensors are clear, find the blur width (sigma) and offset (dy, dx) that make blurred S2 best match
MODIS: search coarsely, then refine. Days without a pair use the median offset of the 3 nearest fitted
days (this recovers MODIS's systematic error; the daily random part stays as noise).
Then **shift MODIS back** by (−dy, −dx).

In [ ]:
PAIR_MIN_CLEAR = 0.95

def _sim_coarse(padded, p, dy, dx):
    win = padded[p - dy:p - dy + HC * K, p - dx:p - dx + WC * K]
    return win.reshape(HC, K, WC, K).mean(axis=(1, 3))

def fit_psf_and_shift(fine, coarse, sigmas=(0, 4, 8, 12, 16, 20, 24), max_shift=20):
    """fine: S2 band (H, W) on the 10 m grid. coarse: MODIS band (HC, WC) on the common grid."""
    p = max_shift
    nanblk = np.isnan(fine).reshape(HC, K, WC, K).any(axis=(1, 3))
    m = np.isfinite(coarse) & ~nanblk
    best = dict(r=-np.inf)
    for s in sigmas:
        f = fill_nan(fine)
        g = np.pad(gaussian_filter(f, s, mode='nearest') if s else f, p, mode='edge')
        search = [(dy, dx) for dy in range(-p, p + 1, 4) for dx in range(-p, p + 1, 4)]
        for stage in range(2):
            for dy, dx in search:
                sim = _sim_coarse(g, p, dy, dx)
                r = np.corrcoef(sim[m], coarse[m])[0, 1]
                if r > best['r']:
                    best = dict(sigma=s, dy=dy, dx=dx, r=r)
            if stage == 0 and best.get('sigma') == s:   # refine around the best coarse hit
                search = [(best['dy'] + a, best['dx'] + b) for a in range(-3, 4) for b in range(-3, 4)
                          if abs(best['dy'] + a) <= p and abs(best['dx'] + b) <= p]
            else:
                break
    return best

pair_days = [d for d in S2_DAYS if s2_clear_frac[d] >= PAIR_MIN_CLEAR and modis_valid[d] >= PAIR_MIN_CLEAR]
psf_fit = {}
for d in pair_days:
    psf_fit[d] = fit_psf_and_shift(s2[d][1], modis_common[d][1])
# answer key: S2 (aligned to the master date) must move by -(MODIS error) - (master's own shift)
true_off = {d: -MODIS_TRUE[d]['err'] - S2_TRUE_SHIFT[MASTER] for d in psf_fit}
fit_tab = pd.DataFrame([dict(day=d, sigma=f['sigma'], dy=f['dy'], dx=f['dx'], r=round(f['r'], 3),
                             true_dy=round(true_off[d][0], 1), true_dx=round(true_off[d][1], 1))
                        for d, f in psf_fit.items()])
print(f'{len(pair_days)} clear same-day pairs. Fitted offsets vs the simulator\'s true offsets:')
display(fit_tab)

fitted = np.array(sorted(psf_fit))
def offset_for_day(d):
    if d in psf_fit:
        return psf_fit[d]['dy'], psf_fit[d]['dx']
    near = fitted[np.argsort(np.abs(fitted - d))[:3]]
    return (float(np.median([psf_fit[n]['dy'] for n in near])), float(np.median([psf_fit[n]['dx'] for n in near])))

def align_modis(up, d):
    dy, dx = offset_for_day(d)
    valid = np.isfinite(up[0]).astype(np.float32)
    out = np.stack([nd_shift(np.nan_to_num(b), (-dy, -dx), order=1, mode='nearest') for b in up])
    out[:, nd_shift(valid, (-dy, -dx), order=1, mode='constant') < 0.999] = np.nan
    return out

### 2.7 · Band adjustment (per band: a·C + b)
MODIS and S2 bands have different spectral responses. On the clear pairs, degrade S2 with each day's
fitted blur and offset, then regress it on MODIS: `S2_degraded = a·MODIS + b`, per band, pooled across
days. Apply (a, b) to all MODIS data. The simulator's true answer is `a = 1/g, b = −o/g`.

In [ ]:
def degrade_to_common(fine, sigma, dy, dx):
    g = np.stack([gaussian_filter(fill_nan(b), sigma, mode='nearest') if sigma else fill_nan(b) for b in fine])
    g = nd_shift(g, (0, dy, dx), order=1, mode='nearest')
    blk = g.reshape(2, HC, K, WC, K).mean(axis=(2, 4))
    blk[:, np.isnan(fine[0]).reshape(HC, K, WC, K).any(axis=(1, 3))] = np.nan
    return blk

BAND_ADJ = []
for b in range(2):
    xs, ys = [], []
    for d, f in psf_fit.items():
        deg = degrade_to_common(s2[d], f['sigma'], f['dy'], f['dx'])
        ok = np.isfinite(deg[b]) & np.isfinite(modis_common[d][b])
        xs.append(modis_common[d][b][ok]); ys.append(deg[b][ok])
    a, c = np.polyfit(np.concatenate(xs), np.concatenate(ys), 1)
    BAND_ADJ.append((a, c))
    print(f'{BANDS[b]:>4}: a = {a:.3f} (true {1 / MODIS_GAIN[b]:.3f}),  b = {c:+.4f} (true {-MODIS_OFFSET[b] / MODIS_GAIN[b]:+.4f})')

def band_adjust(x):
    return np.stack([a * x[i] + c for i, (a, c) in enumerate(BAND_ADJ)]).astype(np.float32)

### 2.8 · Output of Module 2
* `s2[d]`: cloud-masked, co-registered Sentinel-2 reflectance (2, 750, 750), NaN = invalid.
* `modis_10m(d)`: MODIS reflectance **resampled → shifted back → band-adjusted**, (2, 750, 750).

Both are reflectance (red, NIR). **NDVI is not computed in this module.**
The check below shows what preprocessing bought us: how well MODIS agrees with S2 (degraded to MODIS
scale) before and after the corrections.

In [ ]:
_modis_cache = {}
def modis_10m(d):
    if d not in _modis_cache:
        _modis_cache[d] = band_adjust(align_modis(modis_upsampled_10m(d), d))
    return _modis_cache[d]

def agreement(d, corrected):
    m = modis_10m(d) if corrected else modis_upsampled_10m(d)
    blk = lambda a: a.reshape(2, HC, K, WC, K).mean(axis=(2, 4))
    ref = blk(np.stack([gaussian_filter(fill_nan(b), 10) for b in s2[d]]))
    mm = blk(m)
    ok = np.isfinite(mm[1]) & np.isfinite(ref[1])
    return np.sqrt(np.mean((mm[:, ok] - ref[:, ok]) ** 2)), np.corrcoef(mm[1][ok], ref[1][ok])[0, 1]

before = np.array([agreement(d, False) for d in pair_days])
after = np.array([agreement(d, True) for d in pair_days])
print(f'MODIS vs S2 at 250 m over {len(pair_days)} clear pairs')
print(f'  before preprocessing: RMSE {before[:, 0].mean():.4f}, NIR r {before[:, 1].mean():.3f}')
print(f'  after  preprocessing: RMSE {after[:, 0].mean():.4f}, NIR r {after[:, 1].mean():.3f}')

---
# Module 3 · Dataset
Turns preprocessed images into training samples **(C1, F1, C2) → F2**:
* **Clear dates:** S2 ≥ 95 % clear *and* MODIS ≥ 95 % valid on the same day.
* **Split by date, not by patch:** some clear dates are held out entirely for testing (Module 8) and
  validation. Random patch splits leak, because neighbouring patches of one date are nearly identical.
* **Pairs:** any two training dates with |t2 − t1| ≤ 32 days, in both directions.
* **Patches:** 256 × 256 px, stride 128, skipped if any of the four images is > 5 % invalid; random
  flips and 90° rotations applied identically to all four.

In [ ]:
MAX_GAP, PATCH, STRIDE = 32, 256, 128
clear_dates = [d for d in S2_DAYS if s2_clear_frac[d] >= 0.95 and modis_valid[d] >= 0.95]
inner_dates = clear_dates[1:-1]                        # keep the first/last clear dates for training
TEST_DATES = inner_dates[1::3]
VAL_DATES = [d for d in inner_dates[::3] if d not in TEST_DATES][:2]
TRAIN_DATES = [d for d in clear_dates if d not in TEST_DATES + VAL_DATES]
print('clear dates:', clear_dates)
print('train:', TRAIN_DATES, '| val:', VAL_DATES, '| test (held out):', TEST_DATES)

def make_pairs(targets, refs):
    return [(t1, t2) for t2 in targets for t1 in refs if t1 != t2 and abs(t2 - t1) <= MAX_GAP]

class FusionPatches(torch.utils.data.Dataset):
    def __init__(self, pairs, augment=True, max_nan=0.05):
        self.augment, self.items = augment, []
        for t1, t2 in pairs:
            stack = [modis_10m(t1), s2[t1], modis_10m(t2), s2[t2]]
            for i in range(0, H - PATCH + 1, STRIDE):
                for j in range(0, W - PATCH + 1, STRIDE):
                    worst = max(np.isnan(a[0, i:i + PATCH, j:j + PATCH]).mean() for a in stack)
                    if worst <= max_nan:
                        self.items.append((t1, t2, i, j))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, k):
        t1, t2, i, j = self.items[k]
        sl = (slice(None), slice(i, i + PATCH), slice(j, j + PATCH))
        c1, f1, c2, f2 = (modis_10m(t1)[sl], s2[t1][sl], modis_10m(t2)[sl], s2[t2][sl])
        f1 = np.where(np.isnan(f1), c1, f1)            # rare gaps: fall back to the MODIS value
        f2 = np.where(np.isnan(f2), c2, f2)
        x = torch.from_numpy(np.nan_to_num(np.stack([c1, f1, c2, f2]))).clamp(0, 1)
        if self.augment:
            if torch.rand(1) < 0.5: x = x.flip(-1)
            if torch.rand(1) < 0.5: x = x.flip(-2)
            x = torch.rot90(x, int(torch.randint(0, 4, (1,))), dims=(-2, -1))
        return x[0], x[1], x[2], x[3]

train_ds = FusionPatches(make_pairs(TRAIN_DATES, TRAIN_DATES))
val_ds = FusionPatches(make_pairs(VAL_DATES, TRAIN_DATES), augment=False)
print(len(train_ds), 'training patches |', len(val_ds), 'validation patches')

---
# Module 4 · Fusion network (reflectance in → reflectance out)
EDCSTFN-style network (Paper 1 in your comparison table), as drawn:
* **Fine encoder:** extracts field detail from F1.
* **Change encoder:** learns the *fine-scale* change from [C1, F1, C2]. Its last layer is linear because
  change can be negative.
* **Fusion:** adds the two feature maps.
* **Decoder:** maps features back to red and NIR reflectance → **predicted image F̂2**.

*Optional* (dashed note in the diagram): a second reference pair after t2, weighted by how little MODIS
changed. `forward` supports it, and later upgrades slot in here (dilated/U-Net change encoder,
attention fusion, or GAN-STFM / SwinSTFM / STFDiff).

In [ ]:
def conv(cin, cout, k=3):
    return nn.Sequential(nn.ReflectionPad2d(k // 2), nn.Conv2d(cin, cout, k))

class FineEncoder(nn.Module):
    def __init__(self, bands, ch=(32, 64, 128)):
        super().__init__()
        layers, c = [], bands
        for o in ch:
            layers += [conv(c, o), nn.ReLU(inplace=True)]
            c = o
        self.net = nn.Sequential(*layers)
    def forward(self, f1):
        return self.net(f1)

class ChangeEncoder(nn.Module):
    def __init__(self, bands, ch=(32, 64, 128)):
        super().__init__()
        layers, c = [], bands * 3
        for i, o in enumerate(ch):
            layers.append(conv(c, o))
            if i < len(ch) - 1:
                layers.append(nn.ReLU(inplace=True))
            c = o
        self.net = nn.Sequential(*layers)
    def forward(self, c1, f1, c2):
        return self.net(torch.cat([c1, f1, c2], dim=1))

class Decoder(nn.Module):
    def __init__(self, bands, ch=(128, 64, 32)):
        super().__init__()
        self.net = nn.Sequential(conv(ch[0], ch[1]), nn.ReLU(inplace=True),
                                 conv(ch[1], ch[2]), nn.ReLU(inplace=True), nn.Conv2d(ch[2], bands, 1))
    def forward(self, x):
        return self.net(x)

class FusionNet(nn.Module):
    def __init__(self, bands=2):
        super().__init__()
        self.fine_enc, self.change_enc, self.decoder = FineEncoder(bands), ChangeEncoder(bands), Decoder(bands)
    def predict_one(self, c1, f1, c2):
        return self.fine_enc(f1) + self.change_enc(c1, f1, c2)          # Fusion: add features
    def forward(self, c1, f1, c2, c3=None, f3=None):
        z1 = self.predict_one(c1, f1, c2)
        if c3 is None:
            return self.decoder(z1)
        z3 = self.predict_one(c3, f3, c2)                                # optional 2nd reference
        w1 = 1.0 / ((c2 - c1).abs().mean(1, keepdim=True) + 1e-4)
        w3 = 1.0 / ((c2 - c3).abs().mean(1, keepdim=True) + 1e-4)
        return self.decoder((w1 * z1 + w3 * z3) / (w1 + w3))

model = FusionNet(bands=2).to(DEV)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

---
# Modules 5–6 · Loss & training
**Loss** on the predicted red and NIR bands: L1 + 0.8 × (1 − MS-SSIM), plus an **NDVI term computed
from the predicted bands**. That term is only a training signal: it makes the network care about the
band ratio that matters downstream. The NDVI maps themselves are still made last, in Module 9.

**Training:** Adam, learning rate 1e-3 (Paper 1's setting), halved when validation stops improving.
The real F2 (red dashed arrow in the diagram) is the target. Best and last checkpoints are saved.

In [ ]:
def ndvi_t(x):
    return (x[:, 1] - x[:, 0]) / (x[:, 1] + x[:, 0] + 1e-6)

class FusionLoss(nn.Module):
    def __init__(self, w_ssim=0.8, w_ndvi=0.5):
        super().__init__()
        self.w_ssim, self.w_ndvi = w_ssim, w_ndvi
    def forward(self, pred, target):
        l_pix = F.l1_loss(pred, target)
        l_ssim = 1 - ms_ssim(pred.clamp(0, 1), target.clamp(0, 1), data_range=1.0)
        l_ndvi = F.l1_loss(ndvi_t(pred), ndvi_t(target))
        return l_pix + self.w_ssim * l_ssim + self.w_ndvi * l_ndvi

EPOCHS = int(os.environ.get('AGRIVISION_EPOCHS', 2 if FAST else 40))
BATCH = 8
loss_fn = FusionLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=4)
use_amp = DEV == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=BATCH)
if FAST:
    train_dl = torch.utils.data.DataLoader(torch.utils.data.Subset(train_ds, range(min(32, len(train_ds)))),
                                           batch_size=BATCH, shuffle=True, drop_last=True)

history, best = [], float('inf')
t0 = time.time()
for epoch in range(EPOCHS):
    model.train(); tr = 0
    for c1, f1, c2, f2 in train_dl:
        c1, f1, c2, f2 = (t.to(DEV) for t in (c1, f1, c2, f2))
        opt.zero_grad()
        with torch.autocast(DEV, dtype=torch.float16, enabled=use_amp):
            pred = model(c1, f1, c2)
        loss = loss_fn(pred.float(), f2)              # loss in fp32 (MS-SSIM is sensitive)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tr += loss.item()
    model.eval(); va = 0
    with torch.no_grad():
        for c1, f1, c2, f2 in val_dl:
            va += F.l1_loss(model(c1.to(DEV), f1.to(DEV), c2.to(DEV)), f2.to(DEV)).item()
    tr, va = tr / max(len(train_dl), 1), va / max(len(val_dl), 1)
    sched.step(va)
    history.append((tr, va))
    torch.save(model.state_dict(), f'{CKPT_DIR}/last.pt')
    if va < best:
        best = va; torch.save(model.state_dict(), f'{CKPT_DIR}/best.pt')
    print(f'epoch {epoch + 1:2d}  train loss {tr:.4f}  val L1 {va:.4f}  ({time.time() - t0:.0f}s)')

model.load_state_dict(torch.load(f'{CKPT_DIR}/best.pt', map_location=DEV))
plt.plot([h[0] for h in history], label='train loss'); plt.plot([h[1] for h in history], label='val L1')
plt.xlabel('epoch'); plt.legend(); plt.show()

---
# Module 7 · Inference
For **each MODIS-only day t2** (no usable S2): take the **nearest clear S2 date** as reference (F1, C1)
plus that day's MODIS (C2), and predict the full scene in **256 px tiles with 32 px overlap** (only tile
centres are kept, so there are no seams).

The reference pool is train + validation dates. Test dates are left out so Module 8 can score honestly;
in production you would add them back.

In [ ]:
POOL = sorted(TRAIN_DATES + VAL_DATES)
MIN_MODIS_VALID = 0.90

def predict_full(c1, f1, c2, tile=256, pad=32):
    step = tile - 2 * pad
    Hp, Wp = -(-H // step) * step + 2 * pad, -(-W // step) * step + 2 * pad
    padf = lambda a: np.pad(np.nan_to_num(a), ((0, 0), (pad, Hp - H - pad), (pad, Wp - W - pad)), mode='reflect')
    x = [torch.from_numpy(padf(a)).float()[None].to(DEV) for a in (c1, f1, c2)]
    out = np.zeros((2, Hp - 2 * pad, Wp - 2 * pad), np.float32)
    model.eval()
    with torch.no_grad():
        for i in range(0, Hp - 2 * pad, step):
            for j in range(0, Wp - 2 * pad, step):
                y = model(*[t[..., i:i + tile, j:j + tile] for t in x])
                out[:, i:i + step, j:j + step] = y[0, :, pad:pad + step, pad:pad + step].float().cpu().numpy()
    return out[:, :H, :W].clip(0, 1)

def nearest_ref(d, pool=POOL):
    return min(pool, key=lambda r: (abs(r - d), r > d))

def fusion_predict(d, pool=POOL):
    t1 = nearest_ref(d, pool)
    f1 = np.where(np.isnan(s2[t1]), modis_10m(t1), s2[t1])
    out = predict_full(modis_10m(t1), f1, modis_10m(d))
    out[:, np.isnan(modis_10m(d)[0])] = np.nan          # no MODIS → no prediction there
    return out

INFER_DAYS = [d for d in range(N_DAYS) if d not in POOL and modis_valid[d] >= MIN_MODIS_VALID]
GAP_DAYS = [d for d in range(N_DAYS) if d not in POOL and modis_valid[d] < MIN_MODIS_VALID]
if FAST:                                               # smoke test only: a subset of days
    INFER_DAYS = sorted(set(INFER_DAYS[::6]) | set(TEST_DATES))
print(f'{len(INFER_DAYS)} MODIS-only days to predict, {len(POOL)} real S2 days, '
      f'{len(GAP_DAYS)} days with neither (cloudy or off-nadir MODIS)')

### Optional · per-band blend (SOWCEM idea, on reflectance)
The dashed box in the diagram. For each band, blend two guesses per pixel:
* **time guess** (`S`): linear interpolation of the clear S2 reflectance before and after the day;
* **model output** (`T`): the fusion network's prediction.

Weights `w_S + w_T = 1, w ≥ 0` minimise the combined squared error, as in SOWCEM, using the 2×2 error
matrix of the two guesses. The errors are measured on the **validation dates**, which the network never
trained on; using training dates would favour the network unfairly. With only a few dates per pixel, we
smooth the error matrix spatially (our addition) so the weights are stable.

In [ ]:
USE_BLEND = True

def time_guess(d, pool=POOL):
    before = [r for r in pool if r <= d]; after = [r for r in pool if r >= d]
    if not before or not after:
        r = nearest_ref(d, pool)
        return np.where(np.isnan(s2[r]), modis_10m(r), s2[r])
    a, b = before[-1], after[0]
    fa = np.where(np.isnan(s2[a]), modis_10m(a), s2[a]); fb = np.where(np.isnan(s2[b]), modis_10m(b), s2[b])
    if a == b:
        return fa
    w = (d - a) / (b - a)
    return (1 - w) * fa + w * fb

eSS = np.zeros((2, H, W)); eTT = np.zeros((2, H, W)); eST = np.zeros((2, H, W))
for v in VAL_DATES:
    pool_wo = [r for r in POOL if r != v]
    real = s2[v]
    e_s = np.nan_to_num(real - time_guess(v, pool_wo))
    e_t = np.nan_to_num(real - fusion_predict(v, pool_wo))
    eSS += e_s ** 2; eTT += e_t ** 2; eST += e_s * e_t
sm = lambda a: np.stack([gaussian_filter(b, 5) for b in a])
eSS, eTT, eST = sm(eSS), sm(eTT), sm(eST)
# closed-form minimiser of [wS wT] Er [wS wT]^T with wS + wT = 1, clipped to [0, 1] (exact for 2 weights)
W_S = np.clip((eTT - eST) / (eSS + eTT - 2 * eST + 1e-12), 0, 1).astype(np.float32)
W_T = 1 - W_S
print('mean weight on time guess:', dict(zip(BANDS, W_S.mean(axis=(1, 2)).round(2))),
      '| on model:', dict(zip(BANDS, W_T.mean(axis=(1, 2)).round(2))))

def blend(d, model_out, pool=POOL):
    return W_S * time_guess(d, pool) + W_T * model_out

### Predicted Sentinel-2-like image (red + NIR, 10 m, GeoTIFF, every MODIS day)
Each predicted day is written as a georeferenced GeoTIFF (EPSG:32643, 10 m), reflectance × 10 000 as
int16, like the S2 input. These files are the hand-off to Modules 8 and 9.

In [ ]:
def write_geotiff(path, refl, d, source):
    data = np.where(np.isfinite(refl), np.round(refl * 1e4), -9999).astype(np.int16)
    with rasterio.open(path, 'w', driver='GTiff', height=H, width=W, count=2, dtype='int16', crs=CRS,
                       transform=FINE_TRANSFORM, nodata=-9999, compress='deflate') as dst:
        dst.write(data)
        dst.set_band_description(1, 'red'); dst.set_band_description(2, 'nir')
        dst.update_tags(date=str(day2date(d)), source=source)

model_only = {}
t0 = time.time()
for d in INFER_DAYS:
    out = fusion_predict(d)
    if d in TEST_DATES:
        model_only[d] = out
    final = blend(d, out) if USE_BLEND else out
    final[:, np.isnan(out[0])] = np.nan
    write_geotiff(f'{PRED_DIR}/pred_{day2date(d):%Y%m%d}.tif', final, d,
                  'fusion+blend' if USE_BLEND else 'fusion')
print(f'wrote {len(INFER_DAYS)} GeoTIFFs to {PRED_DIR} ({time.time() - t0:.0f}s)')

def read_prediction(d):
    with rasterio.open(f'{PRED_DIR}/pred_{day2date(d):%Y%m%d}.tif') as src:
        a = src.read().astype(np.float32)
    a[a == -9999] = np.nan
    return a / 1e4

---
# Module 8 · Evaluation (held-out real S2 dates)
On every test date we pretend S2 is missing, compare the **predicted image** with the **real S2** image:
1. **Image metrics** on red and NIR: RMSE, SSIM, SAM (spectral angle), ERGAS.
2. **NDVI from the predicted image vs NDVI from the real image**: RMSE and R².

Baselines (from the diagram): **reuse F1** (nearest clear S2 image), **ESTARFM**, and **time
interpolation**. The fusion model must beat them to be worth having.

### ESTARFM baseline
Vectorised ESTARFM (Zhu et al. 2010), run on the GPU when available. For every pixel it searches
**similar pixels** in a moving window on both reference dates, weights them by spectral similarity and
distance, regresses fine on coarse reflectance to get the **conversion coefficient V**, predicts from each
reference date, and blends the two predictions by how much the coarse image changed.
One simplification for speed: candidate pixels are sampled every `step` pixels inside the window.

In [ ]:
def estarfm(F1, C1, F2, C2, Cp, win=51, step=5, n_class=4):
    """F1, C1 / F2, C2: fine & coarse (2, H, W) at two reference dates; Cp: coarse at prediction date."""
    T = lambda a: torch.from_numpy(np.nan_to_num(a)).float().to(DEV)
    F1, C1, F2, C2, Cp = map(T, (F1, C1, F2, C2, Cp))
    r = win // 2
    Fs, Cs = torch.cat([F1, F2]), torch.cat([C1, C2])                     # (4, H, W)
    thr = (2 * Fs.flatten(1).std(1) / n_class).view(-1, 1, 1)
    fm, cm = Fs - Fs.mean(0), Cs - Cs.mean(0)
    R = (fm * cm).sum(0) / (fm.norm(dim=0) * cm.norm(dim=0) + 1e-9)       # fine/coarse spectral correlation
    pad = lambda x: F.pad(x[None], (r, r, r, r), mode='reflect')[0]
    P = {k: pad(v) for k, v in dict(Fs=Fs, Cs=Cs, R=R[None], d1=Cp - C1, d2=Cp - C2).items()}
    wsum = torch.zeros(1, H, W, device=DEV); acc1 = torch.zeros(2, H, W, device=DEV)
    acc2 = torch.zeros(2, H, W, device=DEV)
    n = torch.zeros(2, H, W, device=DEV); sx = torch.zeros_like(n); sy = torch.zeros_like(n)
    sxy = torch.zeros_like(n); sxx = torch.zeros_like(n)
    offs = range(-r, r + 1, step)
    for oy in offs:
        for ox in offs:
            sl = (slice(None), slice(r + oy, r + oy + H), slice(r + ox, r + ox + W))
            Fj = P['Fs'][sl]
            similar = ((Fj - Fs).abs() <= thr).all(0, keepdim=True).float()
            dist = 1 + math.hypot(oy, ox) / r
            w = similar / ((1 - P['R'][sl]).clamp(min=1e-3) * dist)
            wsum += w
            acc1 += w * P['d1'][sl]; acc2 += w * P['d2'][sl]
            Cj = P['Cs'][sl]
            for k in (0, 2):                                              # both reference dates
                x, y = Cj[k:k + 2], Fj[k:k + 2]
                n += similar; sx += similar * x; sy += similar * y
                sxy += similar * x * y; sxx += similar * x * x
    var = sxx - sx ** 2 / n.clamp(min=1)
    V = torch.where((n >= 6) & (var > 1e-6), (sxy - sx * sy / n.clamp(min=1)) / var.clamp(min=1e-9),
                    torch.ones_like(var)).clamp(0.2, 3)
    pred1 = F1 + V * acc1 / wsum.clamp(min=1e-9)
    pred2 = F2 + V * acc2 / wsum.clamp(min=1e-9)
    box = lambda x: F.avg_pool2d(x[None], win, 1, r, count_include_pad=False)[0].sum(0, keepdim=True)
    t1 = 1 / ((box(C1) - box(Cp)).abs() + 1e-6); t2 = 1 / ((box(C2) - box(Cp)).abs() + 1e-6)
    return ((t1 * pred1 + t2 * pred2) / (t1 + t2)).clamp(0, 1).cpu().numpy()

In [ ]:
def ndvi(x):
    return (x[1] - x[0]) / (x[1] + x[0] + 1e-6)

def metrics(pred, real):
    ok = np.isfinite(real[0]) & np.isfinite(pred[0])
    p, t = pred[:, ok].astype(np.float64), real[:, ok].astype(np.float64)
    rmse = np.sqrt(((p - t) ** 2).mean())
    cos = (p * t).sum(0) / (np.linalg.norm(p, axis=0) * np.linalg.norm(t, axis=0) + 1e-12)
    sam = np.degrees(np.arccos(np.clip(cos, -1, 1))).mean()
    band_rmse = np.sqrt(((p - t) ** 2).mean(axis=1))
    ergas = 100 / K * np.sqrt(((band_rmse / t.mean(axis=1)) ** 2).mean())
    ssim = np.mean([structural_similarity(np.where(ok, pred[b], np.nan_to_num(real[b])),
                                          np.nan_to_num(real[b]), data_range=1.0) for b in range(2)])
    np_, nt = ndvi(pred)[ok], ndvi(real)[ok]
    r2 = 1 - ((np_ - nt) ** 2).sum() / ((nt - nt.mean()) ** 2).sum()
    return dict(RMSE=rmse, SSIM=ssim, SAM=sam, ERGAS=ergas,
                NDVI_RMSE=np.sqrt(((np_ - nt) ** 2).mean()), NDVI_R2=r2)

rows = []
for d in TEST_DATES:
    real = s2[d]
    a = max([r for r in POOL if r < d]); b = min([r for r in POOL if r > d])
    fa = np.where(np.isnan(s2[a]), modis_10m(a), s2[a]); fb = np.where(np.isnan(s2[b]), modis_10m(b), s2[b])
    methods = {
        'Fusion model': model_only[d],
        'Fusion + blend (final GeoTIFF)': read_prediction(d),
        'Reuse F1 (nearest clear S2)': np.where(np.isnan(s2[nearest_ref(d)]), np.nan, s2[nearest_ref(d)]),
        'Time interpolation': time_guess(d),
        'ESTARFM': estarfm(fa, modis_10m(a), fb, modis_10m(b), modis_10m(d)),
    }
    for name, pred in methods.items():
        rows.append(dict(date=day2date(d), method=name, **metrics(pred, real)))
results = pd.DataFrame(rows)
summary = results.groupby('method')[['RMSE', 'SSIM', 'SAM', 'ERGAS', 'NDVI_RMSE', 'NDVI_R2']].mean()
summary = summary.sort_values('NDVI_RMSE').round(4)
print(f'Mean over {len(TEST_DATES)} held-out dates {TEST_DATES}:')
display(summary)

In [ ]:
d = TEST_DATES[len(TEST_DATES) // 2]
show = {'Real S2 (held out)': s2[d], 'Fusion + blend': read_prediction(d), 'Fusion model': model_only[d],
        'Reuse F1': s2[nearest_ref(d)], 'Upsampled MODIS': modis_10m(d)}
fig, axs = plt.subplots(1, len(show), figsize=(4 * len(show), 4.2))
for ax, (name, img) in zip(axs, show.items()):
    ax.imshow(ndvi(img)[200:500, 200:500], vmin=0, vmax=0.9, cmap='RdYlGn'); ax.set_title(name); ax.axis('off')
fig.suptitle(f'NDVI made from each image, test date {day2date(d)} (zoom 3 × 3 km)'); plt.show()

---
# Module 9 · NDVI & crop monitoring (LAST STEP)
Everything up to here worked on reflectance. Only now do we compute NDVI, **from the predicted
images** (GeoTIFFs from Module 7) on fused days and from real S2 on real days.

### 9.1 · NDVI = (NIR − Red) / (NIR + Red), from the predicted image
### 9.2 · Time-series cube (real + fused days, fused days flagged)
One layer per day of the season. `source` records which days are real, fused, or gaps.

In [ ]:
cube = np.full((N_DAYS, H, W), np.nan, np.float32)
source = np.array(['gap'] * N_DAYS, dtype=object)
for d in range(N_DAYS):
    if d in POOL:
        cube[d], source[d] = ndvi(s2[d]), 'real'
    elif d in INFER_DAYS:
        cube[d], source[d] = ndvi(read_prediction(d)), 'fused'
print(pd.Series(source).value_counts().to_dict())

### 9.3 · Smoothing (Savitzky–Golay, per pixel)
First fill gaps (cloudy pixels, gap days) by linear interpolation in time, then smooth each pixel's
series with a Savitzky–Golay filter (window 15 days, order 2) to remove day-to-day MODIS noise.
Done in row blocks to keep memory low.

In [ ]:
def fill_and_smooth(block):
    T, h, w = block.shape
    flat = block.reshape(T, -1)
    t = np.arange(T)
    for k in range(flat.shape[1]):
        v = flat[:, k]; ok = np.isfinite(v)
        if ok.any() and not ok.all():
            flat[:, k] = np.interp(t, t[ok], v[ok])
    # pixels never observed at all (rare): use that day's mean over the block
    day_mean = np.nan_to_num(np.nanmean(flat, axis=1, keepdims=True))
    flat = np.where(np.isfinite(flat), flat, day_mean)
    return savgol_filter(flat, 15, 2, axis=0).reshape(T, h, w)

smooth = np.empty_like(cube)
for i in range(0, H, 50):
    smooth[:, i:i + 50] = fill_and_smooth(cube[:, i:i + 50].copy())

### 9.4 · Field statistics & alerts
Per field (from the field boundary layer): mean NDVI each day, then:
* **Anomaly alert (stress):** during the crop's peak period, the field's **own** NDVI falls more than
  0.05 below its best value of the previous 20 days for 5+ days, **while its crop's median is not
  falling**. Comparing a field with its own recent level (rather than with other fields) means a field
  that is simply weaker, or sown a few days late, is not flagged; only a real decline is.
* **Variability (patchiness):** within-field NDVI spread (CV) at mid-season, marked if > 1.5× its crop's
  median. Informational: uneven fields are worth a look, but patchiness alone is not an alert.
* **Phenology:** green-up (crossing half-amplitude going up), peak date and value, senescence.

Because the data are synthetic we can check the alerts against the stress we injected in Module 1.

In [ ]:
fid = field_id.ravel(); okf = fid >= 0
cnt = np.bincount(fid[okf], minlength=N_FIELDS)
field_mean = np.stack([np.bincount(fid[okf], smooth[d].ravel()[okf], N_FIELDS) / np.maximum(cnt, 1)
                       for d in range(N_DAYS)], axis=1)
field_sq = np.stack([np.bincount(fid[okf], smooth[d].ravel()[okf] ** 2, N_FIELDS) / np.maximum(cnt, 1)
                     for d in range(N_DAYS)], axis=1)
field_cv = np.sqrt(np.maximum(field_sq - field_mean ** 2, 0)) / np.maximum(field_mean, 1e-3)

rows = []
crops = [c for c in ('wheat', 'mustard') if (field_class == c).any()]
for c in crops:
    ids = np.flatnonzero((field_class == c) & (cnt > 500))
    med = np.median(field_mean[ids], axis=0)
    cv_ref = np.median(field_cv[ids, 90:110].mean(1))
    for f in ids:
        x = field_mean[f]
        own_drop = np.array([x[t] - x[max(0, t - 20):t + 1].max() for t in range(N_DAYS)])
        crop_drop = np.array([med[t] - med[max(0, t - 20):t + 1].max() for t in range(N_DAYS)])
        low = ((own_drop < -0.05) & (crop_drop > -0.02) & (med >= 0.9 * med.max())).astype(int)
        run = np.convolve(low, np.ones(5, int), 'valid')
        alert_day = int(np.argmax(run >= 5)) if (run >= 5).any() else None
        cv_mid = field_cv[f, 90:110].mean()
        curve = field_mean[f]; amp_lo, amp_hi = curve.min(), curve.max(); half = amp_lo + 0.5 * (amp_hi - amp_lo)
        peak = int(curve.argmax())
        up = int(np.argmax(curve[:peak + 1] >= half)) if peak > 0 else 0
        down = peak + int(np.argmax(curve[peak:] < half)) if (curve[peak:] < half).any() else None
        rows.append(dict(field=f, crop=c, pixels=int(cnt[f]),
                         green_up=str(day2date(up)), peak=str(day2date(peak)), peak_ndvi=round(curve.max(), 2),
                         senescence=str(day2date(down)) if down is not None else '-',
                         anomaly_alert=str(day2date(alert_day)) if alert_day is not None else '',
                         patchy=bool(cv_mid > 1.5 * cv_ref)))
report = pd.DataFrame(rows).set_index('field')
flagged = report[report.anomaly_alert != '']
print('Fields with stress alerts:')
display(flagged)
print('Patchy fields (informational):', list(report.index[report.patchy]))

truth_ids = {s['field'] for s in STRESS}
print('\nInjected stress (answer key):', [(s['field'], str(day2date(s['start_day']))) for s in STRESS])
print('found:', sorted(truth_ids & set(flagged.index)), '| missed:', sorted(truth_ids - set(flagged.index)),
      '| false alarms:', sorted(set(flagged.index) - truth_ids))

### Maps and field curves
Left: smoothed NDVI late in the season with the within-field **hotspot** map (pixels > 0.15 below their
field's mean). Right: the season curve of a stressed field and a healthy one, with real and fused days
marked separately.

In [ ]:
d_map = min(N_DAYS - 1, max(s['start_day'] for s in STRESS) + 20)
fm = field_mean[np.clip(field_id, 0, None), d_map]
hot = (smooth[d_map] < fm - 0.15) & (field_id >= 0)
fig, axs = plt.subplots(1, 3, figsize=(18, 5.5))
axs[0].imshow(smooth[d_map], vmin=0, vmax=0.9, cmap='RdYlGn'); axs[0].set_title(f'Smoothed NDVI {day2date(d_map)}')
axs[1].imshow(hot, cmap='Reds'); axs[1].contour(stress_mask, [0.5], colors='blue', linewidths=0.8)
axs[1].set_title('Hotspots (red) vs injected stress (blue outline)')
for a in axs[:2]: a.axis('off')
s_f = STRESS[0]['field']
ok_f = [f for f in report.index if f not in truth_ids and report.loc[f, 'crop'] == 'wheat']
for f, lbl in [(s_f, f'stressed field {s_f}'), (ok_f[0], f'healthy field {ok_f[0]}')]:
    m = field_id == f
    axs[2].plot(days, field_mean[f], label=lbl)
    for src, mk in (('real', 'o'), ('fused', '.')):
        dd = [d for d in range(N_DAYS) if source[d] == src]
        vals = [cube[d][m][np.isfinite(cube[d][m])] for d in dd]
        axs[2].plot(dd, [v.mean() if v.size else np.nan for v in vals], mk, ms=4, alpha=0.6)
axs[2].axvline(STRESS[0]['start_day'], ls='--', c='gray'); axs[2].set_xlabel('day of season')
axs[2].set_ylabel('field mean NDVI'); axs[2].legend(); axs[2].set_title('o = real S2 day, · = fused day')
plt.show()

---
## What to do when the real data arrives
1. Replace **Module 1** with the GEE exports (guide, M1). Keep the same names: `s2_raw[d]` with
   `B4, B8, cs_cdf`; `mod09gq[d]`, `mod09ga[d]` on their native grids.
2. In **2.2 and 2.5**, use real reprojection (`rasterio.warp.reproject`, bilinear) because real MODIS is
   sinusoidal; in **2.3**, switch to AROSICS `COREG_LOCAL`.
3. Everything from Module 3 on runs unchanged. The answer-key checks (true shifts, injected stress)
   disappear, since real data has no answer key. Module 8's held-out dates become your evidence.